# Занятие 6. Трансформеры в компьютерном зрении (ViT)

## Цели занятия:
- Изучить архитектуру Vision Transformer (ViT)
- Реализовать механизм Patch Embedding и Positional Encoding
- Обучить ViT на CIFAR-10 и сравнить с CNN
- Визуализировать карты внимания (Attention Maps)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import random
import math

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Шаг 1. Подготовка данных (CIFAR-10)

In [ ]:
# ViT requires fixed input size
# Resize CIFAR-10 from 32x32 to 64x64 for better patch representation

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

---
## Шаг 2. Реализация Patch Embedding

In [ ]:
class PatchEmbedding(nn.Module):
    """Convert image into patches and embed them.
    
    Args:
        img_size: Size of input image (square)
        patch_size: Size of each patch
        in_channels: Number of input channels
        embed_dim: Embedding dimension
    """
    
    def __init__(self, img_size=64, patch_size=8, in_channels=3, embed_dim=128):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        
        # Use Conv2d to extract patches and project to embed_dim
        self.projection = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
    
    def forward(self, x):
        # x: (B, C, H, W)
        x = self.projection(x)  # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)  # (B, embed_dim, n_patches)
        x = x.transpose(1, 2)  # (B, n_patches, embed_dim)
        return x

# Test Patch Embedding
patch_embed = PatchEmbedding(img_size=64, patch_size=8, in_channels=3, embed_dim=128)
test_input = torch.randn(1, 3, 64, 64)
test_output = patch_embed(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Number of patches: {patch_embed.n_patches}")

---
## Шаг 3. Сборка модели ViT

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self Attention."""
    
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = self.head_dim ** -0.5
    
    def forward(self, x):
        B, N, C = x.shape
        
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        
        return x, attn  # Return attention for visualization

In [ ]:
class TransformerBlock(nn.Module):
    """Transformer Encoder Block."""
    
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.attn(x_norm)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, attn_weights

In [ ]:
class VisionTransformer(nn.Module):
    """Vision Transformer for image classification.
    
    Args:
        img_size: Input image size
        patch_size: Patch size
        in_channels: Number of input channels
        num_classes: Number of output classes
        embed_dim: Embedding dimension
        depth: Number of transformer blocks
        num_heads: Number of attention heads
    """
    
    def __init__(self, img_size=64, patch_size=8, in_channels=3, num_classes=10,
                 embed_dim=128, depth=4, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        
        # Patch Embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        
        # CLS token and Position Embedding
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)
        
        # Transformer Blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        
        # Store attention weights for visualization
        self.attention_weights = []
    
    def forward(self, x):
        B = x.shape[0]
        
        # Patch Embedding
        x = self.patch_embed(x)  # (B, n_patches, embed_dim)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)  # (B, n_patches+1, embed_dim)
        
        # Add Position Embedding
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        # Transformer Blocks
        self.attention_weights = []
        for block in self.blocks:
            x, attn = block(x)
            self.attention_weights.append(attn)
        
        # Classification head (use CLS token)
        x = self.norm(x)
        cls_output = x[:, 0]  # CLS token
        
        return self.head(cls_output)

# Create model
model = VisionTransformer(
    img_size=64, patch_size=8, in_channels=3, num_classes=10,
    embed_dim=128, depth=4, num_heads=4
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"ViT parameters: {total_params:,}")
print(model)

---
## Шаг 4. Обучение модели

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return running_loss / len(loader), 100 * correct / total

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return 100 * correct / total

In [ ]:
# Training
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

print("="*50)
print("Training Vision Transformer")
print("="*50)

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_acc = evaluate(model, test_loader, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%")

---
## Шаг 5. Визуализация Attention Maps

In [ ]:
import matplotlib.pyplot as plt

# Get a sample image
model.eval()
images, labels = next(iter(test_loader))
sample_img = images[0:1].to(device)

with torch.no_grad():
    output = model(sample_img)
    attn_weights = model.attention_weights  # List of attention tensors

# Visualize attention from first block
# Attention shape: (B, num_heads, n_patches+1, n_patches+1)
attn = attn_weights[0][0].cpu()  # First batch, all heads

print(f"Attention shape: {attn.shape}")

# CLS token attention to patches
cls_attn = attn[:, 0, 1:]  # CLS token's attention to patches (excluding itself)

# Reshape to spatial grid
patch_size = 8
grid_size = 64 // patch_size  # 8x8

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

# Show original image
img_np = sample_img[0].cpu().numpy()
img_np = img_np.transpose(1, 2, 0) * 0.5 + 0.5  # Denormalize
axes[0, 0].imshow(img_np)
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

# Show attention maps for each head
for i, ax in enumerate(axes.flat[1:]):
    if i < cls_attn.shape[0]:
        attn_map = cls_attn[i].reshape(grid_size, grid_size)
        im = ax.imshow(attn_map, cmap='jet')
        ax.set_title(f'Head {i+1} Attention')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig('vit_attention.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'])
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vit_training.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Домашнее задание

1. Изменить размер патча (4x4 vs 8x8) и сравнить точность
2. Добавить данные аугментации (CutMix или MixUp)
3. Сравнить количество параметров ViT и ResNet18

---
## Контрольные вопросы

1. Зачем нужен токен [CLS] в ViT?
2. Почему ViT требует больше данных для обучения чем CNN?
3. Как вычисляется размерность последовательности патчей?
4. В чем преимущество Self-Attention перед свертками?